In [56]:
import os
import re
import string
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/guilherme/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/guilherme/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [57]:
import spacy

nlp = spacy.load("pt_core_news_sm")

def lematizar_texto(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]

In [ ]:
def limpar_e_preparar_dados(caminhos_ficheiros):
    """
    Lê os ficheiros, remove pontuação, converte para minúsculas 
    e tokeniza em frases/palavras.
    """
    sentencas_processadas = []
    stop_words = set(stopwords.words('portuguese'))

    translator = str.maketrans('', '', string.punctuation)
    
    for caminho in caminhos_ficheiros:
        try:    
            print(f"A processar: {caminho}...")
            with open(caminho, 'r', encoding='utf-8') as f:
                for linha in f:
                    # 1. Limpeza básica e remoção de números/pontuação
                    linha = linha.strip().lower()
                    linha = re.sub(r"\d+", "", linha)
                    linha = linha.translate(translator)

                    # 2. Tokenização (dividir em palavras)
                    tokens = word_tokenize(linha)

                    # 3. Eliminar StopWords
                    stopWRemove = [w for w in tokens if w not in stop_words and len(w) > 2]

                    # 4. Lemmatize
                    if stopWRemove:
                        final_clean_text = lematizar_texto(stopWRemove)                    
                        sentencas_processadas.append(final_clean_text)
                        
        except FileNotFoundError:
            print(f"Erro: O ficheiro {caminho} não foi encontardo.")
        except Exception as e:
            print(f"Aviso: Ocorreu um erro: {e}"+
                  f"\nO ficheiro {caminho} não foi encontrado.")
        

    return sentencas_processadas

In [ ]:
meus_ficheiros = ['livros/Harry Potter e A Pedra Filosofal.txt', 'livros/Harry_Potter_Camara_Secreta-br.txt'
                  , 'livros/harry_potter_e_o_calice_de_fogo-J._K._Rowling.txt','livros/harry_potter_e_o_enigma_do_principe-J._K._Rowling.txt',
                  'livros/J.K.Rowling-5-Harry_Potter_e_a_Ordem_da_Fenix.txt', 'livros/J.K.Rowling-7-Harry_Potter_e_As_Reliquias_da_Morte.txt'] 

dados_treino = limpar_e_preparar_dados(meus_ficheiros)

if not dados_treino:
    print("Erro: Não foram encontrados dados para treinar o modelo.")
else:
    print(f"Treinando modelo com {len(dados_treino)} frases...")

    model = Word2Vec(
        sentences=dados_treino,
        vector_size=100,
        epochs=20, 
        window=5, 
        min_count=2, 
        workers=4
    )

    model_name = "models/my_model_word2vec.model"
    model.save(model_name)

A processar: livros/Harry Potter e A Pedra Filosofal.txt...
A processar: livros/Harry_Potter_Camara_Secreta-br.txt...
A processar: livros/harry_potter_e_o_calice_de_fogo-J._K._Rowling.txt...
A processar: livros/harry_potter_e_o_enigma_do_principe-J._K._Rowling.txt...
A processar: livros/J.K.Rowling-5-Harry_Potter_e_a_Ordem_da_Fenix.txt...
A processar: livros/J.K.Rowling-7-Harry_Potter_e_As_Reliquias_da_Morte.txt...
Erro: Não foram encontrados dados para treinar o modelo.


In [68]:
from gensim.models import Word2Vec

def analysis(model):
    print("\n\n______Welcome to Harry Potter Books analysis______\n\n")

    print("#1 Fase:\n")

    for token, score in model.wv.most_similar('hermione'):
        print(f"Token: {token} -> {score:.4f}")

analysis(model)



______Welcome to Harry Potter Books analysis______


#1 Fase:

Token: mione -> 0.6484
Token: Gina -> 0.4873
Token: simas -> 0.4460
Token: apreensivo -> 0.4433
Token: quê -> 0.4394
Token: hermioner -> 0.4344
Token: simo -> 0.4260
Token: lilá -> 0.4233
Token: xenófilo -> 0.4166
Token: nervoso -> 0.4119
